In [24]:
param = {
    "test": "power",
    "sample_size": 200,
    "batch_size": 128,
    "z_dim": 1,
    "dx": 1,
    "dy": 1,
    "n_test": 100,
    "epochs_num": 100,
    "eps_std": 1.0,
    "dist_z": 'gaussian',
    "alpha_x": 0.20,
    "m_value": 100,
    "k_value": 8,
    "j_value": 1000,
    "noise_dimension": 5,
    "noise_dimension_type": "normal",
    "noise_dimension_var": 1,
    "hidden_layer_size": 1024,
    "normal_ini": False,
    "preprocess": 'scale',
    "G_lr": 7e-5,
    "alpha": 0.1,
    "alpha1": 0.05,
    "set_seeds": 0,
    "using_orcale": False,
    "lambda_1": 1,
    "lambda_2": 1,
    "using_Gen": '1',
    "boor_rv_type": 'gaussian',
    "wgt_decay": 1e-5,
    "lambda_3": 1e-4,
    "lambda_4": 2e-5,
    "drop_out_p": 0.2,
    "is_sparse": True,
    "sparse_ratio": 0.05,
    "lambda_mean": 0.5,
    "mean_samples": 20
}

import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools
from tqdm import tqdm

# [修改1] 删除全局 device，改为在 mGAN() 内部按 gpu_id 动态创建


def generate_samples_random(size=1000, sType='H0', dx=1, dy=1, dz=1, nstd=1.0, alpha_x=0.5,
                             preprocess="None", dist_z='gaussian', delta_peak=3.0):
    num = size
    error_generator_x = TD.MultivariateNormal(torch.zeros(dx), torch.eye(dx))
    error_generator_y = TD.MultivariateNormal(torch.zeros(dy), torch.eye(dy))

    if dist_z == 'gaussian':
        z_generator = TD.MultivariateNormal(torch.zeros(dz), torch.eye(dz))
    elif dist_z == 'laplace':
        z_generator = TD.Independent(TD.Laplace(torch.zeros(dz), torch.ones(dz)), 1)
    else:
        z_generator = TD.MultivariateNormal(torch.zeros(dz), torch.eye(dz))

    Z = z_generator.sample((num,))
    error_x = error_generator_x.sample((num,))
    error_y = error_generator_y.sample((num,))

    z_abs_mean = torch.mean(torch.abs(Z), dim=1, keepdim=True)
    sigma_z = 0.3 + 1.2 * z_abs_mean

    if dy == dz:
        Z_for_Y = Z
    elif dy < dz:
        Z_for_Y = Z[:, :dy]
    else:
        pad = torch.zeros(num, dy - dz, dtype=Z.dtype, device=Z.device)
        Z_for_Y = torch.cat([Z, pad], dim=1)

    Y = Z_for_Y + nstd * error_y

    if dx == dy:
        error_y_for_x = error_y
    elif dx < dy:
        error_y_for_x = error_y[:, :dx]
    else:
        pad = torch.zeros(num, dx - dy, dtype=error_y.dtype, device=error_y.device)
        error_y_for_x = torch.cat([error_y, pad], dim=1)

    sigma_x = sigma_z.repeat(1, dx)

    if sType == 'H0':
        X = sigma_x * error_x
    else:
        m = TD.Bernoulli(torch.tensor([alpha_x], dtype=Z.dtype, device=Z.device))
        Delta = m.sample((num,)).view(num, 1)
        shared_noise = (1 - Delta) * error_x + Delta * error_y_for_x
        X = sigma_x * shared_noise

    return X, Y, Z


def generate_samples_from_fixed_Z_random(Z, size=1000, sType='H0', dx=1, dy=1, dz=1, nstd=1.0,
                                          alpha_x=0.05, normalize=True, seed=None,
                                          dist_z='gaussian', delta_peak=3.0):
    if seed is not None:
        torch.manual_seed(seed)

    num = size
    if Z.dim() == 1:
        Z = Z.view(num, 1)

    error_generator_x = TD.MultivariateNormal(torch.zeros(dx), torch.eye(dx))
    error_generator_y = TD.MultivariateNormal(torch.zeros(dy), torch.eye(dy))

    error_x = error_generator_x.sample((num,))
    error_y = error_generator_y.sample((num,))

    z_abs_mean = torch.mean(torch.abs(Z), dim=1, keepdim=True)
    sigma_z = 0.3 + 1.2 * z_abs_mean

    if dy == dz:
        Z_for_Y = Z
    elif dy < dz:
        Z_for_Y = Z[:, :dy]
    else:
        pad = torch.zeros(num, dy - dz, dtype=Z.dtype, device=Z.device)
        Z_for_Y = torch.cat([Z, pad], dim=1)

    Y = Z_for_Y + nstd * error_y

    if dx == dy:
        error_y_for_x = error_y
    elif dx < dy:
        error_y_for_x = error_y[:, :dx]
    else:
        pad = torch.zeros(num, dx - dy, dtype=error_y.dtype, device=error_y.device)
        error_y_for_x = torch.cat([error_y, pad], dim=1)

    sigma_x = sigma_z.repeat(1, dx)

    if sType == 'H0':
        X = sigma_x * error_x
    else:
        m = TD.Bernoulli(torch.tensor([alpha_x], dtype=Z.dtype, device=Z.device))
        Delta = m.sample((num,)).view(num, 1)
        shared_noise = (1 - Delta) * error_x + Delta * error_y_for_x
        X = sigma_x * shared_noise

    return X, Y


def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch,
                        x_torch, y_torch, z_torch,
                        sigma_w, sigma_u=1, sigma_v=1, boor_rv_type="gaussian",
                        device=None):  # [修改2] 新增 device 参数
    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2)
    w_mx = torch.exp(-w_mx / sigma_w)

    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch.repeat(n, 1, 1)
                              - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u), dim=2)
    u_mx_3 = u_mx_2.T

    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)

    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1, n) - x_torch.repeat(1, n).T) / sigma_v)
    v_mx_2 = torch.mean(
        torch.exp(-torch.abs(gen_x_all_torch.repeat(n, 1, 1)
                              - x_torch.repeat(1, n).reshape(n, n, 1)) / sigma_v), dim=2)
    v_mx_3 = v_mx_2.T

    gen_x_all_torch_rep = gen_x_all_torch.repeat(n, 1, 1)
    temp2_mx = gen_x_all_torch_rep[:, :, 0].T
    sum2_mx = torch.mean(
        torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)

    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        sum_mx = sum_mx + torch.mean(
            torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)
        temp2_mx = gen_x_all_torch_rep[:, :, i].T
        sum2_mx = sum2_mx + torch.mean(
            torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)

    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + 1 / M * sum_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + 1 / M * sum2_mx

    FF_mx = u_mx * v_mx * w_mx * (1 - torch.eye(n).to(device))  # [修改2]
    stat = 1 / (n - 1) * torch.sum(FF_mx).item()

    boottemp = np.array([])
    if boor_rv_type == "rademacher":
        eboot = torch.sign(torch.randn(n, boot_num)).to(device)  # [修改2]
    elif boor_rv_type == "gaussian":
        eboot = torch.randn(n, boot_num).to(device)              # [修改2]
    for bb in range(boot_num):
        random_mx = torch.matmul(eboot[:, bb].reshape(-1, 1), eboot[:, bb].reshape(-1, 1).T)
        stat_boot = 1 / (n - 1) * torch.sum(FF_mx * random_mx).item()
        boottemp = np.append(boottemp, stat_boot)

    return stat, boottemp


class DatasetSelect(Dataset):
    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]


class DatasetSelect_GAN(torch.utils.data.Dataset):
    def __init__(self, X, Y, Z, batch_size):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return (self.X_real[index], self.Y_real[index], self.Z_real[index],
                self.Z_real[(self.batch_size + index) % self.sample_size])


class DatasetSelect_GAN_ver2(torch.utils.data.Dataset):
    def __init__(self, Y, Z, batch_size):
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = Z.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.Y_real[index], self.Z_real[index]


def sample_noise(sample_size, noise_dimension, noise_type, input_var=1,
                 device=None):  # [修改3] 新增 device 参数
    if noise_type == "normal":
        noise_generator = TD.MultivariateNormal(
            torch.zeros(noise_dimension).to(device),            # [修改3]
            input_var * torch.eye(noise_dimension).to(device))  # [修改3]
        Z = noise_generator.sample((sample_size,))
    elif noise_type == "unif":
        Z = torch.rand(sample_size, noise_dimension)
    elif noise_type == "Cauchy":
        Z = TD.Cauchy(torch.tensor([0.0]), torch.tensor([1.0])).sample(
            (sample_size, noise_dimension)).squeeze(2)
    else:
        raise ValueError(f"Unknown noise_type: {noise_type}")
    return Z


class Generator(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef, drop_out_p, drop_input=False):
        super(Generator, self).__init__()
        self.BN_type = BN_type
        self.ReLU_coef = ReLU_coef
        self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN3 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
        self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)
        self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc3 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)
        self.sigmoid = torch.nn.Sigmoid()
        self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out2 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out3 = torch.nn.Dropout(p=drop_out_p)
        self.drop_input = drop_input

    def forward(self, x):
        if self.BN_type:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
            x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
            x = self.fc_last(x)
        else:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
            x = self.drop_out2(self.leakyReLU1(self.fc2(x)))
            x = self.fc_last(x)
        return x


class NonFullyConnected_1(torch.nn.Module):
    def __init__(self, size_in, size_out, m, bias=True):
        super(NonFullyConnected_1, self).__init__()
        # [修改3] 不在构造函数中 .to(device)，由外层统一 .to(device)
        self.linear = torch.nn.Linear(m * size_in, m * size_out, bias=bias)
        self.mask = functools.reduce(
            torch.block_diag, [torch.ones(size_out, size_in) for _ in range(m)])

    def forward(self, x):
        # [修改3] mask 动态跟随模型所在设备
        self.linear.weight.data *= self.mask.to(self.linear.weight.device)
        return self.linear(x)


class Generator_2(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef,
                 hidden_layer_depth=1, ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)
        self.linear_layers_from_input = torch.nn.Linear(
            self.input_dimension, ntargets_k * self.hidden_layer_sizes[0])
        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0], self.hidden_layer_sizes[0], ntargets_k)
            for _ in range(len(self.hidden_layer_sizes))
        ])
        self.linear8 = torch.nn.Linear(
            self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

    def forward(self, input):
        if self.BN_type:
            output = self.leakyrelu(self.BN1(self.linear_layers_from_input(input)))
            for layer in self.linear_layers_between:
                output = self.leakyrelu(self.BN1(layer(output)))
        else:
            output = self.leakyrelu(self.linear_layers_from_input(input))
            for layer in self.linear_layers_between:
                output = self.leakyrelu(layer(output))
        return self.linear8(output)


def find_loss(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-torch.abs(Y.repeat(1, n) - Y.repeat(1, n).T) / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    mx_1_2 = torch.exp(-mx_1_2 / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2 = torch.exp(-torch.abs(Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y) * mx_1_2
    mx_3 = mx_2.T
    mx_4 = torch.exp(-torch.abs(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y) * mx_1_2
    return 1 / (n ** 2) * torch.sum(mx_1 - mx_2 - mx_3 + mx_4)


def find_loss_2(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-(Y.repeat(1, n) - Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=2, dim=2)
    mx_1_2 = torch.exp(-(mx_1_2 ** 2) / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2 = torch.exp(-(Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y) * mx_1_2
    mx_3 = mx_2.T
    mx_4 = torch.exp(-(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y) * mx_1_2
    return 1 / (n ** 2) * torch.sum(mx_1 - mx_2 - mx_3 + mx_4)


def train_ver3(
        G_zx, G_zy,
        X, Y, Z, X_test, Y_test, Z_test, M,
        noise_dimension, noise_type, G_lr, hidden_layer_size,
        DataLoader, BN_type, ReLU_coef,
        lambda_mean=0.5, mean_samples=20,
        epochs_num=50,
        patience=5, min_delta=1e-5,
        sigma_z=1, sigma_x=1, sigma_y=1,
        normal_ini=False,
        lambda_1=1, lambda_2=1, using_Gen='1', wgt_decay=0,
        lambda_3=0, drop_out_p=0.2, noise_dimension_var=1,
        lambda_4=0,
        device=None):  # [修改3] 新增 device 参数

    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]

    if G_zy is None or G_zx is None:
        if using_Gen == '1':
            G_zy = Generator(input_dimension, output_dimension_y, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改3]
            G_zx = Generator(input_dimension, output_dimension_x, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改3]
        elif using_Gen == '2':
            G_zy = Generator_2(input_dimension, output_dimension_y, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改3]
            G_zx = Generator_2(input_dimension, output_dimension_x, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改3]

        if normal_ini:
            for p in G_zy.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改3]
            for p in G_zx.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改3]

    G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

    G_zy.train()
    G_zx.train()
    best_loss = float('inf')
    counter = 0

    for epoch in range(epochs_num):
        G_zy.train()
        G_zx.train()
        epoch_loss = 0.0
        num_batches = 0

        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)   # [修改3]
            Y_real = Y_real.to(device)   # [修改3]
            Z_real = Z_real.to(device)   # [修改3]
            Z_fake = Z_fake.to(device)   # [修改3]

            batch_size = Z_real.shape[0]
            Z_repeated = Z_real.repeat_interleave(mean_samples, dim=0)

            Noise_for_mean = sample_noise(
                Z_repeated.shape[0], noise_dimension, noise_type,
                input_var=noise_dimension_var, device=device).to(device)   # [修改3]

            Noise_fake_y = sample_noise(
                Z_real.shape[0], noise_dimension, noise_type,
                input_var=noise_dimension_var, device=device).to(device)   # [修改3]

            Noise_fake_x = sample_noise(
                Z_real.shape[0], noise_dimension, noise_type,
                input_var=noise_dimension_var, device=device).to(device)   # [修改3]

            Y_fake = G_zy(torch.cat((Z_real, Noise_fake_y), dim=1))
            X_fake = G_zx(torch.cat((Z_real, Noise_fake_x), dim=1))

            # mean alignment — G_zy
            Y_generated_group = G_zy(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            Y_mean_pred = torch.mean(
                Y_generated_group.reshape(batch_size, mean_samples, -1), dim=1)
            loss_mean_y = torch.nn.functional.mse_loss(Y_mean_pred, Y_real)

            # mean alignment — G_zx
            X_generated_group = G_zx(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            X_mean_pred = torch.mean(
                X_generated_group.reshape(batch_size, mean_samples, -1), dim=1)
            loss_mean_x = torch.nn.functional.mse_loss(X_mean_pred, X_real)

            # train G_zy
            G_zy_solver.zero_grad()
            l1_first, l1_rest = 0.0, 0.0
            for name, param in G_zy.named_parameters():
                if "fc1" in name:
                    l1_first += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_rest += torch.linalg.vector_norm(param, ord=1)
            mmd_loss_y = (lambda_1 * find_loss(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y)
                          + lambda_2 * find_loss_2(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y)
                          + lambda_3 * l1_rest + lambda_4 * l1_first)
            g_zy_error = mmd_loss_y + lambda_mean * loss_mean_y
            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()

            # train G_zx
            G_zx_solver.zero_grad()
            l1_first, l1_rest = 0.0, 0.0
            for name, param in G_zx.named_parameters():
                if "fc1" in name:
                    l1_first += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_rest += torch.linalg.vector_norm(param, ord=1)
            mmd_loss_x = (lambda_1 * find_loss(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x)
                          + lambda_2 * find_loss_2(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x)
                          + lambda_3 * l1_rest + lambda_4 * l1_first)
            g_zx_error = mmd_loss_x + lambda_mean * loss_mean_x
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()

            epoch_loss += g_zy_error.item() + g_zx_error.item()
            num_batches += 1

        current_loss = epoch_loss / max(num_batches, 1)
        if current_loss < best_loss - min_delta:
            best_loss = current_loss
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            break

    return G_zy, G_zx


def mGAN(n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=1000,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=500, k=2, boot_num=1000,
         noise_dimension=10, hidden_layer_size=512, normal_ini=False, preprocess='normalize',
         G_lr=1e-5, using_orcale=False, lambda_1=1, lambda_2=1,
         lambda_mean=0.3, mean_samples=20,
         using_Gen='1',
         boor_rv_type="gaussian", wgt_decay=0, lambda_3=1, drop_out_p=0.2,
         noise_dimension_var=1, noise_dimension_type="normal", lambda_4=1,
         gpu_id=0):  # [修改4] 新增 gpu_id 参数

    # [修改4] 在函数内部根据 gpu_id 创建局部 device
    device = torch.device(f'cuda:{gpu_id}' if torch.cuda.is_available() else 'cpu')

    if simulation == 'type1error':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n, sType='H0', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess)
    elif simulation == 'power':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n, sType='H1', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess)
    else:
        raise ValueError('Test does not exist.')

    x, y, z = sim_x.to(device), sim_y.to(device), sim_z.to(device)

    w_mx = torch.linalg.vector_norm(
        z.repeat(n, 1, 1) - torch.swapaxes(z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    sigma_w = torch.median(w_mx).item()
    sigma_u = torch.median(torch.abs(y.repeat(1, n) - y.repeat(1, n).T)).item()
    sigma_v = torch.median(torch.abs(x.repeat(1, n) - x.repeat(1, n).T)).item()

    test_size = int(n / k)
    stat_all = torch.zeros(k, 1)
    boot_temp_all = torch.zeros(k, boot_num)
    cur_k = 0

    if not using_orcale:
        G_zy = Generator(z_dim, y_dims, noise_dimension, hidden_layer_size,
                         False, 0.1, drop_out_p).to(device)   # [修改4]
        G_zx = Generator(z_dim, x_dims, noise_dimension, hidden_layer_size,
                         False, 0.1, drop_out_p).to(device)   # [修改4]

    for k_fold in range(k):
        k_fold_start = int(n / k * k_fold)
        k_fold_end = int(n / k * (k_fold + 1))

        X_test = x[k_fold_start:k_fold_end]
        Y_test = y[k_fold_start:k_fold_end]
        Z_test = z[k_fold_start:k_fold_end]
        X_train = torch.cat((x[:k_fold_start], x[k_fold_end:]))
        Y_train = torch.cat((y[:k_fold_start], y[k_fold_end:]))
        Z_train = torch.cat((z[:k_fold_start], z[k_fold_end:]))
        if k == 1:
            X_train, Y_train, Z_train = X_test, Y_test, Z_test

        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)
        DataLoader_xyz = torch.utils.data.DataLoader(
            train_xyz, batch_size=batch_size, shuffle=True)

        if not using_orcale:
            current_epochs = epochs_num if k_fold == 0 else max(10, epochs_num // 5)
            G_zy, G_zx = train_ver3(
                G_zx=G_zx, G_zy=G_zy,
                X=X_train, Y=Y_train, Z=Z_train, M=M,
                X_test=X_test, Y_test=Y_test, Z_test=Z_test,
                noise_dimension=noise_dimension, noise_type=noise_dimension_type,
                G_lr=G_lr, hidden_layer_size=hidden_layer_size,
                DataLoader=DataLoader_xyz, BN_type=False, ReLU_coef=0.1,
                lambda_mean=lambda_mean, mean_samples=mean_samples,
                epochs_num=current_epochs,
                sigma_z=sigma_w, sigma_x=sigma_v, sigma_y=sigma_u,
                normal_ini=normal_ini, lambda_1=lambda_1, lambda_2=lambda_2,
                using_Gen=using_Gen, wgt_decay=wgt_decay, lambda_3=lambda_3,
                drop_out_p=drop_out_p, noise_dimension_var=noise_dimension_var,
                lambda_4=lambda_4,
                device=device)  # [修改4] 传入局部 device

        dataset_test = DatasetSelect(X_test, Y_test, Z_test)
        dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=True)

        gen_x_all = torch.zeros(test_size, M, device=device)    # [修改4]
        gen_y_all = torch.zeros(test_size, M, device=device)    # [修改4]
        z_all = torch.zeros(test_size, z_dim, device=device)    # [修改4]
        x_all = torch.zeros(test_size, x_dims, device=device)   # [修改4]
        y_all = torch.zeros(test_size, y_dims, device=device)   # [修改4]
        cur_itr = 0

        if not using_orcale:
            G_zx.eval()
            G_zy.eval()

        for i, (x_test, y_test, z_test) in enumerate(dataloader_test):
            z_test_temp = z_test.repeat(M, 1)
            if not using_orcale:
                z_test_temp = z_test_temp.to(device)                              # [修改4]
                Noise_fake_x = sample_noise(
                    z_test_temp.size(0), noise_dimension, noise_dimension_type,
                    input_var=noise_dimension_var, device=device).to(device)      # [修改4]
                with torch.no_grad():
                    fake_x = G_zx(torch.cat((z_test_temp, Noise_fake_x), dim=1)).reshape(1, -1)
                Noise_fake_y = sample_noise(
                    z_test_temp.size(0), noise_dimension, noise_dimension_type,
                    input_var=noise_dimension_var, device=device).to(device)      # [修改4]
                with torch.no_grad():
                    fake_y = G_zy(torch.cat((z_test_temp, Noise_fake_y), dim=1)).reshape(1, -1)
            elif using_orcale:
                stype = 'H0' if simulation == 'type1error' else 'H1'
                fake_x, fake_y = generate_samples_from_fixed_Z_random(
                    z_test_temp, size=M, sType=stype,
                    dx=x_dims, dy=y_dims, dz=z_dim, nstd=nstd,
                    alpha_x=a_x, dist_z=z_dist)

            gen_x_all[cur_itr, :] = fake_x.detach().reshape(-1)
            gen_y_all[cur_itr, :] = fake_y.detach().reshape(-1)
            x_all[cur_itr, :] = x_test
            y_all[cur_itr, :] = y_test
            z_all[cur_itr, :] = z_test
            cur_itr += 1

        cur_stat, cur_boot_temp = get_p_value_stat_1(
            boot_num, M, test_size,
            gen_x_all.to(device), gen_y_all.to(device),
            x_all.to(device), y_all.to(device), z_all.to(device),
            sigma_w, sigma_u, sigma_v, boor_rv_type,
            device=device)  # [修改4]

        stat_all[cur_k, :] = cur_stat
        boot_temp_all[cur_k, :] = torch.from_numpy(cur_boot_temp)
        cur_k += 1

        if using_orcale:
            gen_x_mean = torch.mean(gen_x_all, dim=1).reshape(-1, 1)
            gen_y_mean = torch.mean(gen_y_all, dim=1).reshape(-1, 1)
            mse_x = torch.mean((gen_x_mean - x_all) ** 2).item()
            mse_y = torch.mean((gen_y_mean - y_all) ** 2).item()
            print(f'Test MSE x [{mse_x}], MSE y [{mse_y}]')

    return np.mean(torch.mean(boot_temp_all, dim=0).numpy() > torch.mean(stat_all).item())


# ============================================================
# 并行部分
# ============================================================
from joblib import Parallel, delayed
import multiprocessing

def run_experiment(params):
    test                 = params["test"]
    sample_size          = params["sample_size"]
    batch_size           = params["batch_size"]
    z_dim                = params["z_dim"]
    dx                   = params["dx"]
    dy                   = params["dy"]
    n_test               = params["n_test"]
    epochs_num           = params["epochs_num"]
    eps_std              = params["eps_std"]
    dist_z               = params["dist_z"]
    alpha_x              = params["alpha_x"]
    m_value              = params["m_value"]
    k_value              = params["k_value"]
    j_value              = params["j_value"]
    noise_dimension      = params["noise_dimension"]
    noise_dimension_type = params["noise_dimension_type"]
    noise_dimension_var  = params["noise_dimension_var"]
    hidden_layer_size    = params["hidden_layer_size"]
    normal_ini           = params["normal_ini"]
    preprocess           = params["preprocess"]
    G_lr                 = params["G_lr"]
    alpha                = params["alpha"]
    alpha1               = params["alpha1"]
    set_seeds            = params["set_seeds"]
    using_orcale         = params["using_orcale"]
    lambda_1             = params["lambda_1"]
    lambda_2             = params["lambda_2"]
    using_Gen            = params["using_Gen"]
    boor_rv_type         = params["boor_rv_type"]
    wgt_decay            = params["wgt_decay"]
    lambda_3             = params["lambda_3"]
    lambda_4             = params["lambda_4"]
    drop_out_p           = params["drop_out_p"]
    lambda_mean          = params["lambda_mean"]
    mean_samples         = params["mean_samples"]

    # [修改4] 获取可用 GPU 数量，轮询分配
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    max_workers_per_gpu = 2                               # 每块 GPU 最多同时跑 2 个 worker，可按显存调整
    num_cores = min(28, n_test)
    if num_cores < 1:
        num_cores = 1

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 开始并行实验...")
    print(f"模式: {test}, 样本量: {sample_size}, 交叉验证折数: {k_value}, "
          f"均值对齐系数: {lambda_mean}, 实验次数: {n_test}, "
          f"并行核数: {num_cores}, 可用GPU数: {num_gpus}")
    if test == 'power':
        print(f"备择假设H_1下的模型参数 alpha_x: {alpha_x}")

    def run_one_trial(trial_id):
        trial_seed = set_seeds + trial_id
        np.random.seed(trial_seed)
        torch.manual_seed(trial_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(trial_seed)
            torch.cuda.manual_seed_all(trial_seed)

        return mGAN(
            n=sample_size, z_dim=z_dim, simulation=test, batch_size=batch_size,
            epochs_num=epochs_num, nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
            a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
            noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
            normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
            using_orcale=using_orcale, lambda_1=lambda_1, lambda_2=lambda_2,
            lambda_mean=lambda_mean, mean_samples=mean_samples,
            using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
            lambda_3=lambda_3, drop_out_p=drop_out_p,
            noise_dimension_var=noise_dimension_var,
            noise_dimension_type=noise_dimension_type, lambda_4=lambda_4,
            gpu_id=trial_id % num_gpus  # [修改4] 轮询分配 GPU
        )

    # [修改4] enumerate 获取 trial_id 用于轮询
    p_values = Parallel(n_jobs=num_cores)(
        delayed(run_one_trial)(trial_id) for trial_id in range(n_test)
    )

    p_values = np.array(p_values)
    fp = [pval < alpha for pval in p_values]
    final_result = np.mean(fp)
    fp1 = [pval < alpha1 for pval in p_values]
    final_result1 = np.mean(fp1)

    print("\n" + "=" * 50)
    print(f"实验类型: {test.upper()}")
    print(f"Z Dimension: {z_dim}")
    print(f"Emp Rej Rate: {final_result:.4f} (at significance level {alpha})")
    print(f"Emp Rej Rate: {final_result1:.4f} (at significance level {alpha1})")
    print("=" * 50 + "\n")

    return p_values


In [19]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[15:39:49] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1500 (at significance level 0.1)
Emp Rej Rate: 0.0850 (at significance level 0.05)

[15:43:18] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1450 (at significance level 0.0548)
Emp Rej Rate: 0.0900 (at significance level 0.031)

[15:46:25] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3150 (at significance level 0.0548)
Emp Rej Rate: 0.2550 (at significance level 0.031)

[15:49:37] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.6000 (at significance level 0.0548)
Emp Rej Rate: 0.4900 (at significance level 0.031)

[15

In [21]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0.5
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[16:00:38] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1950 (at significance level 0.1)
Emp Rej Rate: 0.1300 (at significance level 0.05)

[16:03:41] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1700 (at significance level 0.043800000000000006)
Emp Rej Rate: 0.1050 (at significance level 0.022)

[16:06:46] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3450 (at significance level 0.043800000000000006)
Emp Rej Rate: 0.2400 (at significance level 0.022)

[16:09:50] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.6200 (at significance level 0.043800000000000006)
Emp R

In [23]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0.3
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[16:19:28] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.3, 实验次数: 200, 并行核数: 32, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1850 (at significance level 0.1)
Emp Rej Rate: 0.1100 (at significance level 0.05)

[16:22:58] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.3, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1300 (at significance level 0.047900000000000005)
Emp Rej Rate: 0.1050 (at significance level 0.02695)

[16:26:05] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.3, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2900 (at significance level 0.047900000000000005)
Emp Rej Rate: 0.2300 (at significance level 0.02695)

[16:29:15] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.3, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5750 (at significance level 0.047900000000000005)
E

In [25]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0.2
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[16:39:30] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.2, 实验次数: 200, 并行核数: 28, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1750 (at significance level 0.1)
Emp Rej Rate: 0.0950 (at significance level 0.05)

[16:42:59] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.2, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1350 (at significance level 0.051800000000000006)
Emp Rej Rate: 0.1000 (at significance level 0.0299)

[16:46:07] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.2, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3000 (at significance level 0.051800000000000006)
Emp Rej Rate: 0.2250 (at significance level 0.0299)

[16:49:13] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.2, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5650 (at significance level 0.051800000000000006)
Emp

In [2]:
param["sample_size"] = 500
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0.5
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[13:46:18] 开始并行实验...
模式: type1error, 样本量: 500, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4


/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.2600 (at significance level 0.1)
Emp Rej Rate: 0.1550 (at significance level 0.05)

[13:50:10] 开始并行实验...
模式: power, 样本量: 500, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1800 (at significance level 0.03750000000000001)
Emp Rej Rate: 0.0550 (at significance level 0.006)

[13:53:40] 开始并行实验...
模式: power, 样本量: 500, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.4050 (at significance level 0.03750000000000001)
Emp Rej Rate: 0.1500 (at significance level 0.006)

[13:57:10] 开始并行实验...
模式: power, 样本量: 500, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.6550 (at significance level 0.03750000000000001)
Emp Rej Rate: 0.4550 (at significance level 0.006)

[14:00:41] 开始并行实验...
模式: power, 样本量: 500, 交叉验证折数: 2, 均值对齐系

In [4]:
param["sample_size"] = 500
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[14:08:54] 开始并行实验...
模式: type1error, 样本量: 500, 交叉验证折数: 2, 均值对齐系数: 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1700 (at significance level 0.1)
Emp Rej Rate: 0.0800 (at significance level 0.05)

[14:12:25] 开始并行实验...
模式: power, 样本量: 500, 交叉验证折数: 2, 均值对齐系数: 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05


Process LokyProcess-331:
Process LokyProcess-330:
Process LokyProcess-328:
Process LokyProcess-327:
Process LokyProcess-329:
Process LokyProcess-309:
Process LokyProcess-310:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py", line 528, in _process_worker
    with worker_exit_lock:
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/backend/synchronize.py", line 119, in __enter__
    return self._semlock.acquire()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108,

KeyboardInterrupt: 

In [10]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 300
param["lambda_mean"] = 0.5
param["mean_samples"] = 50
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[14:38:01] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 300, 并行核数: 28, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1800 (at significance level 0.1)
Emp Rej Rate: 0.0900 (at significance level 0.05)

[14:43:20] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 300, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1433 (at significance level 0.0528)
Emp Rej Rate: 0.0767 (at significance level 0.0219)

[14:48:24] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 300, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1


Process LokyProcess-667:
Traceback (most recent call last):
Process LokyProcess-660:
Process LokyProcess-672:
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py", line 528, in _process_worker
    with worker_exit_lock:
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/backend/synchronize.py", line 119, in __enter__
    return self._semlock.acquire()
KeyboardInterrupt
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/23099458d/venv

KeyboardInterrupt: 

In [12]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 400
param["lambda_mean"] = 0.5
param["mean_samples"] = 50
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[14:49:04] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 400, 并行核数: 28, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1950 (at significance level 0.1)
Emp Rej Rate: 0.1000 (at significance level 0.05)

[14:56:06] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 均值对齐系数: 0.5, 实验次数: 400, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05


Process LokyProcess-788:
Process LokyProcess-789:
Process LokyProcess-762:
Process LokyProcess-772:
Process LokyProcess-777:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py", line 528, in _process_worker
    with worker_exit_lock:
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/backend/synchronize.py", line 119, in __enter__
    return self._semlock.acquire()
KeyboardInterrupt
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 1

KeyboardInterrupt: 

In [7]:
# ============================================================
# 参数配置
# ============================================================
param = {
    "test": "power",
    "sample_size": 200,
    "batch_size": 128,
    "z_dim": 1,
    "dx": 1,
    "dy": 1,
    "n_test": 100,
    "epochs_num": 100,
    "eps_std": 1.0,
    "dist_z": 'gaussian',
    "alpha_x": 0.20,
    "m_value": 100,
    "k_value": 8,
    "j_value": 1000,
    "noise_dimension": 5,
    "noise_dimension_type": "normal",
    "noise_dimension_var": 1,
    "hidden_layer_size": 1024,
    "normal_ini": False,
    "preprocess": 'scale',
    "G_lr": 7e-5,
    "alpha": 0.1,
    "alpha1": 0.05,
    "set_seeds": 0,
    "using_orcale": False,
    "lambda_1": 1,
    "lambda_2": 1,
    "using_Gen": '1',
    "boor_rv_type": 'gaussian',
    "wgt_decay": 1e-5,
    "lambda_3": 1e-4,
    "lambda_4": 2e-5,
    "drop_out_p": 0.2,
    "is_sparse": True,
    "sparse_ratio": 0.05,
    "lambda_mean": 0.3,
    "mean_samples": 20
}

import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools
from tqdm import tqdm

# 删除全局 device，改为在 mGAN() 内部按 gpu_id 动态创建


def generate_samples_random(size=1000, sType='H0', dx=1, dy=1, dz=1, nstd=1.0,
                             alpha_x=0.05, preprocess="None", dist_z='gaussian'):
    """
    异方差版本 DGP：
        Z ~ N(0, I)
        sigma(Z) = 0.3 + 1.2 * |Z|          （条件标准差随 Z 变化）

    H0（条件独立）：
        Y = Z + eps_Y,       eps_Y ~ N(0, I)
        X = sigma(Z) * eps_X, eps_X ~ N(0, I)   X 的方差依赖 Z，但均值不依赖 Y

    H1（条件相关）：
        Y 同上
        以概率 alpha_x，X 的噪声替换为 eps_Y（引入与 Y 的相关性）：
        X = sigma(Z) * [(1-delta)*eps_X + delta*eps_Y],  delta ~ Bernoulli(alpha_x)

    关键特性：
        E[X|Z] = 0（均值对齐后为零均值），
        Var[X|Z] = sigma(Z)^2（异方差），
        L3 均值损失仅惩罚 E[X|Z] 的偏差，不影响方差学习。
    """
    num = size
    error_generator_x = TD.MultivariateNormal(torch.zeros(dx), 1 * torch.eye(dx))
    error_generator_y = TD.MultivariateNormal(torch.zeros(dy), 1 * torch.eye(dy))

    if dist_z == 'gaussian':
        z_generator = TD.MultivariateNormal(torch.zeros(dz), 1 * torch.eye(dz))
        Z = z_generator.sample((num,))
    elif dist_z == 'laplace':
        z_generator = TD.MultivariateNormal(torch.zeros(dz), 1 * torch.eye(dz))
        Z = z_generator.sample((num,))

    error_x = error_generator_x.sample((num,))
    error_y = error_generator_y.sample((num,))

    # 异方差标准差：sigma(Z) = 0.3 + 1.2 * |Z|，shape (num, 1)
    sigma_z = 0.3 + 1.2 * torch.mean(torch.abs(Z), dim=1, keepdim=True)  # (num, 1)
    sigma_x = sigma_z.repeat(1, dx)  # (num, dx)

    # Y 保持同方差（与原版一致，便于与原始实验对比）
    Y = Z[:, :dy] + nstd * error_y  # 取 Z 的前 dy 维

    # 用于 X 中混合噪声的 error_y（维度对齐到 dx）
    if dx == dy:
        error_y_for_x = error_y
    elif dx < dy:
        error_y_for_x = error_y[:, :dx]
    else:
        pad = torch.zeros(num, dx - dy)
        error_y_for_x = torch.cat([error_y, pad], dim=1)

    if sType == 'H0':
        # H0：X = sigma(Z) * eps_X，与 Y 条件独立
        X = sigma_x * error_x
    else:
        # H1：以概率 alpha_x 将噪声替换为 eps_Y，引入条件相关性
        m = TD.Bernoulli(torch.tensor([alpha_x]))
        delta = m.sample((num,))                             # (num, 1)
        shared_noise = (1 - delta) * error_x + delta * error_y_for_x
        X = sigma_x * shared_noise

    return X, Y, Z


def generate_samples_from_fixed_Z_random(Z, size=1000, sType='H0', dx=1, dy=1, dz=1,
                                          nstd=1.0, alpha_x=0.05, normalize=True,
                                          seed=None, dist_z='gaussian'):
    """
    异方差版本（固定 Z 采样），用于 oracle 模式。
    """
    num = size
    if Z.dim() == 1:
        Z = Z.view(num, dz)

    error_generator_x = TD.MultivariateNormal(torch.zeros(dx), 1 * torch.eye(dx))
    error_generator_y = TD.MultivariateNormal(torch.zeros(dy), 1 * torch.eye(dy))

    error_x = error_generator_x.sample((num,))
    error_y = error_generator_y.sample((num,))

    sigma_z = 0.3 + 1.2 * torch.mean(torch.abs(Z), dim=1, keepdim=True)
    sigma_x = sigma_z.repeat(1, dx)

    Y = Z[:, :dy] + nstd * error_y

    if dx == dy:
        error_y_for_x = error_y
    elif dx < dy:
        error_y_for_x = error_y[:, :dx]
    else:
        pad = torch.zeros(num, dx - dy)
        error_y_for_x = torch.cat([error_y, pad], dim=1)

    if sType == 'H0':
        X = sigma_x * error_x
    else:
        m = TD.Bernoulli(torch.tensor([alpha_x]))
        delta = m.sample((num,))
        shared_noise = (1 - delta) * error_x + delta * error_y_for_x
        X = sigma_x * shared_noise

    return X, Y
# ============================================================
# [修改结束]
# ============================================================


def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch,
                        x_torch, y_torch, z_torch, sigma_w,
                        sigma_u=1, sigma_v=1, boor_rv_type="gaussian",
                        device=None):
    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1),
        ord=1, dim=2)
    w_mx = torch.exp(-w_mx / sigma_w)
    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(torch.exp(-torch.abs(
        gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u), dim=2)
    u_mx_3 = u_mx_2.T
    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(torch.exp(-torch.abs(
        gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)
    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1, n) - x_torch.repeat(1, n).T) / sigma_v)
    v_mx_2 = torch.mean(torch.exp(-torch.abs(
        gen_x_all_torch.repeat(n, 1, 1) - x_torch.repeat(1, n).reshape(n, n, 1)) / sigma_v), dim=2)
    v_mx_3 = v_mx_2.T
    gen_x_all_torch_rep = gen_x_all_torch.repeat(n, 1, 1)
    temp2_mx = gen_x_all_torch_rep[:, :, 0].T
    sum2_mx = torch.mean(torch.exp(-torch.abs(
        gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)
    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(torch.exp(-torch.abs(
            gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)
        sum_mx = sum_mx + temp_add_mx
        temp2_mx = gen_x_all_torch_rep[:, :, i].T
        temp2_add_mx = torch.mean(torch.exp(-torch.abs(
            gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)
        sum2_mx = sum2_mx + temp2_add_mx
    u_mx_4 = 1 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4
    v_mx_4 = 1 / M * sum2_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + v_mx_4
    FF_mx = u_mx * v_mx * w_mx * (1 - torch.eye(n).to(device))
    stat = 1 / (n - 1) * torch.sum(FF_mx).item()
    boottemp = np.array([])
    if boor_rv_type == "rademacher":
        eboot = torch.sign(torch.randn(n, boot_num)).to(device)
    elif boor_rv_type == "gaussian":
        eboot = torch.randn(n, boot_num).to(device)
    for bb in range(boot_num):
        random_mx = torch.matmul(eboot[:, bb].reshape(-1, 1), eboot[:, bb].reshape(-1, 1).T)
        bootmatrix = FF_mx * random_mx
        stat_boot = 1 / (n - 1) * torch.sum(bootmatrix).item()
        boottemp = np.append(boottemp, stat_boot)
    return stat, boottemp


class DatasetSelect(Dataset):
    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]


class DatasetSelect_GAN(torch.utils.data.Dataset):
    def __init__(self, X, Y, Z, batch_size):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return (self.X_real[index], self.Y_real[index], self.Z_real[index],
                self.Z_real[(self.batch_size + index) % self.sample_size])


class DatasetSelect_GAN_ver2(torch.utils.data.Dataset):
    def __init__(self, Y, Z, batch_size):
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = Z.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.Y_real[index], self.Z_real[index]


##### Auxilliary functions #####

def sample_noise(sample_size, noise_dimension, noise_type, input_var=1, device=None):
    if noise_type == "normal":
        noise_generator = TD.MultivariateNormal(
            torch.zeros(noise_dimension).to(device),
            input_var * torch.eye(noise_dimension).to(device))
        Z = noise_generator.sample((sample_size,))
    if noise_type == "unif":
        Z = torch.rand(sample_size, noise_dimension)
    if noise_type == "Cauchy":
        Z = TD.Cauchy(torch.tensor([0.0]), torch.tensor([1.0])).sample(
            (sample_size, noise_dimension)).squeeze(2)
    return Z


##### GAN architecture #####

class Generator(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef, drop_out_p, drop_input=False):
        super(Generator, self).__init__()
        self.BN_type = BN_type
        self.ReLU_coef = ReLU_coef
        self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN3 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
        self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)
        self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc3 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)
        self.sigmoid = torch.nn.Sigmoid()
        self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out2 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out3 = torch.nn.Dropout(p=drop_out_p)
        self.drop_input = drop_input

    def forward(self, x):
        if self.BN_type:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
            x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
            x = self.fc_last(x)
        else:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
            x = self.drop_out2(self.leakyReLU1(self.fc2(x)))
            x = self.fc_last(x)
        return x


class NonFullyConnected_1(torch.nn.Module):
    def __init__(self, size_in, size_out, m, bias=True):
        super(NonFullyConnected_1, self).__init__()
        self.linear = torch.nn.Linear(m * size_in, m * size_out, bias=bias)
        self.mask = functools.reduce(
            torch.block_diag,
            [torch.ones(size_out, size_in) for i in range(m)])

    def forward(self, x):
        self.linear.weight.data *= self.mask.to(self.linear.weight.device)
        return self.linear(x)


class Generator_2(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension,
                 hidden_layer_size, BN_type, ReLU_coef,
                 hidden_layer_depth=1, ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)
        self.linear_layers_from_input = torch.nn.Linear(
            self.input_dimension, ntargets_k * self.hidden_layer_sizes[0])
        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0], self.hidden_layer_sizes[0], ntargets_k)
            for i in range(len(self.hidden_layer_sizes))
        ])
        self.linear8 = torch.nn.Linear(
            self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

    def forward(self, input):
        if self.BN_type:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(self.BN1(output))
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(self.BN1(output))
        else:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(output)
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(output)
        return self.linear8(output)


##### Training procedures #####

def find_loss(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-torch.abs(Y.repeat(1, n) - Y.repeat(1, n).T) / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    mx_1_2 = torch.exp(-mx_1_2 / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-torch.abs(Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-torch.abs(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss


def find_loss_2(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-(Y.repeat(1, n) - Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=2, dim=2)
    mx_1_2 = torch.exp(-(mx_1_2 ** 2) / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-(Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss


def compute_loss_mean_L3(Y_real, Y_generated_group, batch_size, mean_samples):
    """
    计算 L3 (U-statistic) mean alignment loss。
    使用 Note.pdf Section 5 中的等价计算形式：

        L3 = (1/n) * sum_i [ M/(M-1) * ||Y_i - G_bar_M(X_i)||^2
                             - 1/(M*(M-1)) * sum_m ||Y_i - G(X_i, eps_{i,m})||^2 ]

    其中：
        第一项 = M/(M-1) * L2，是 L2 loss 乘以缩放系数
        第二项 = 负的逐样本 MSE 均值（L1 形式），精确抵消方差坍缩

    参数：
        Y_real:             shape (batch_size, d)，真实样本
        Y_generated_group:  shape (batch_size * mean_samples, d)，M 个生成样本（已展开）
        batch_size:         批大小 n
        mean_samples:       M，每个样本生成的数量
    """
    M = mean_samples

    # reshape 为 (batch_size, M, d)
    Y_gen_reshaped = Y_generated_group.reshape(batch_size, M, -1)  # (n, M, d)

    # G_bar_M(X_i)：对 M 个生成样本取均值，shape (n, d)
    Y_mean_pred = torch.mean(Y_gen_reshaped, dim=1)

    # 第一项：M/(M-1) * ||Y_i - G_bar_M(X_i)||^2，均值化为标量
    # mse_loss 内部已经做了 mean over batch 和 dim
    term1 = (M / (M - 1)) * torch.nn.functional.mse_loss(Y_mean_pred, Y_real)

    # 第二项：1/(M*(M-1)) * sum_m ||Y_i - G(X_i, eps_{i,m})||^2，均值化为标量
    # Y_real 扩展为 (n, M, d) 以便逐样本计算
    Y_real_expanded = Y_real.unsqueeze(1).expand_as(Y_gen_reshaped)  # (n, M, d)
    # 逐元素平方差，sum over d，mean over n 和 M
    per_sample_mse = torch.mean((Y_real_expanded - Y_gen_reshaped) ** 2)
    term2 = (1 / (M - 1)) * per_sample_mse
    # 注意：mse_loss 已经做了 mean over (n*M*d)，等价于
    # (1/(n*M*d)) * sum_{i,m,j} (...)^2
    # 而公式要求的是 (1/(M*(M-1))) * sum_m (1/n)*sum_i ||...||^2
    # = (1/(M-1)) * mean_{i,m}[||...||^2]，与上面等价（d 维的 mse_loss 已包含 mean over d）

    loss_mean = term1 - term2
    return loss_mean


def train_ver3(
        G_zx, G_zy,
        X, Y, Z, X_test, Y_test, Z_test, M,
        noise_dimension, noise_type, G_lr, hidden_layer_size,
        DataLoader, BN_type, ReLU_coef,
        lambda_mean=0.5, mean_samples=20,
        epochs_num=50, patience=5, min_delta=1e-5,
        sigma_z=1, sigma_x=1, sigma_y=1, normal_ini=False,
        lambda_1=1, lambda_2=1, using_Gen='1', wgt_decay=0,
        lambda_3=0, drop_out_p=0.2, noise_dimension_var=1, lambda_4=0,
        device=None):

    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]

    if G_zy is None or G_zx is None:
        if using_Gen == '1':
            G_zy = Generator(input_dimension, output_dimension_y, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)
            G_zx = Generator(input_dimension, output_dimension_x, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)
        elif using_Gen == '2':
            G_zy = Generator_2(input_dimension, output_dimension_y, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)
            G_zx = Generator_2(input_dimension, output_dimension_x, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)
        if normal_ini:
            for p in G_zy.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))
            for p in G_zx.parameters():
                p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))

    G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

    iter_count = 0
    G_zy = G_zy.train()
    G_zx = G_zx.train()

    best_loss = float('inf')
    counter = 0

    for epoch in range(epochs_num):
        batch_count = 0
        G_zy = G_zy.train()
        G_zx = G_zx.train()

        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)
            Y_real = Y_real.to(device)
            Z_real = Z_real.to(device)
            Z_fake = Z_fake.to(device)

            batch_size = Z_real.shape[0]
            # repeat_interleave: Z 变成 [z1,z1,...,z2,z2,...] 共 batch_size*mean_samples 行
            Z_repeated = Z_real.repeat_interleave(mean_samples, dim=0)

            # 生成用于 mean loss 的噪声（共 batch_size * mean_samples 个）
            Noise_for_mean = sample_noise(Z_repeated.shape[0], noise_dimension, noise_type,
                                          input_var=noise_dimension_var, device=device).to(device)

            # 生成用于 MMD loss 的 fake 样本（每个样本 1 个）
            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)
            Y_fake = G_zy(torch.cat((Z_real, Noise_fake), dim=1)).to(device)
            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)
            X_fake = G_zx(torch.cat((Z_real, Noise_fake), dim=1)).to(device)

            # 生成 M 个样本用于 mean loss（shape: batch_size * mean_samples, d）
            Y_generated_group = G_zy(torch.cat((Z_repeated, Noise_for_mean), dim=1))
            X_generated_group = G_zx(torch.cat((Z_repeated, Noise_for_mean), dim=1))

            # ==============================================================
            # [修改] 使用 L3（U-statistic）代替 L2 计算 mean alignment loss
            # 原 L2 代码：
            #   Y_mean_pred = torch.mean(Y_generated_group.reshape(batch_size, mean_samples, -1), dim=1)
            #   loss_mean_y = torch.nn.functional.mse_loss(Y_mean_pred, Y_real)
            #
            # 新 L3 公式（等价计算形式，见 Note.pdf Section 5）：
            #   loss_mean = M/(M-1) * L2_term  -  1/(M-1) * L1_term
            # ==============================================================
            loss_mean_y = compute_loss_mean_L3(Y_real, Y_generated_group, batch_size, mean_samples)
            loss_mean_x = compute_loss_mean_L3(X_real, X_generated_group, batch_size, mean_samples)

            # G_zy 的 MMD loss + L3 mean loss
            g_zy_error = None
            G_zy_solver.zero_grad()
            g_zx_error = None
            G_zx_solver.zero_grad()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param in G_zy.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_y = (lambda_1 * find_loss(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_2 * find_loss_2(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zy_error = mmd_loss_y + lambda_mean * loss_mean_y
            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param in G_zx.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param, ord=1)

            mmd_loss_x = (lambda_1 * find_loss(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_2 * find_loss_2(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zx_error = mmd_loss_x + lambda_mean * loss_mean_x
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()

            iter_count += 1
            batch_count += 1

        current_loss = g_zx_error + g_zy_error
        if current_loss < best_loss - min_delta:
            best_loss = current_loss
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            break

    return G_zy, G_zx


def mGAN(n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=1000,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=500, k=2, boot_num=1000,
         noise_dimension=10, hidden_layer_size=512, normal_ini=False, preprocess='normalize',
         G_lr=1e-5, using_orcale=False, lambda_1=1, lambda_2=1,
         lambda_mean=0.3, mean_samples=20, using_Gen='1',
         boor_rv_type="gaussian", wgt_decay=0, lambda_3=1, drop_out_p=0.2,
         noise_dimension_var=1, noise_dimension_type="normal", lambda_4=1,
         gpu_id=0):
    """
    gpu_id: 该 worker 使用的 GPU 编号，由 run_experiment() 按 job_index % num_gpus 分配。
    """
    enable_cuda = True
    if torch.cuda.is_available() and enable_cuda:
        device = torch.device(f'cuda:{gpu_id}')
    else:
        device = torch.device('cpu')

    if simulation == 'type1error':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n, sType='H0', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess)
    elif simulation == 'power':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n, sType='H1', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess)
    else:
        raise ValueError('Test does not exist.')

    x, y, z = sim_x.to(device), sim_y.to(device), sim_z.to(device)

    w_mx = torch.linalg.vector_norm(
        z.repeat(n, 1, 1) - torch.swapaxes(z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    sigma_w = torch.median(w_mx).item()
    u_mx = torch.exp(-torch.abs(y.repeat(1, n) - y.repeat(1, n).T))
    sigma_u = torch.median(torch.abs(y.repeat(1, n) - y.repeat(1, n).T)).item()
    v_mx = torch.exp(-torch.abs(x.repeat(1, n) - x.repeat(1, n).T))
    sigma_v = torch.median(torch.abs(x.repeat(1, n) - x.repeat(1, n).T)).item()

    test_size = int(n / k)
    stat_all = torch.zeros(k, 1)
    boot_temp_all = torch.zeros(k, boot_num)
    cur_k = 0

    if not using_orcale:
        G_zy = Generator(z_dim, y_dims, noise_dimension, hidden_layer_size,
                         False, 0.1, drop_out_p).to(device)
        G_zx = Generator(z_dim, x_dims, noise_dimension, hidden_layer_size,
                         False, 0.1, drop_out_p).to(device)

    for k_fold in range(k):
        k_fold_start = int(n / k * k_fold)
        k_fold_end = int(n / k * (k_fold + 1))
        X_test = x[k_fold_start:k_fold_end]
        Y_test = y[k_fold_start:k_fold_end]
        Z_test = z[k_fold_start:k_fold_end]
        X_train = torch.cat((x[0:k_fold_start], x[k_fold_end:]))
        Y_train = torch.cat((y[0:k_fold_start], y[k_fold_end:]))
        Z_train = torch.cat((z[0:k_fold_start], z[k_fold_end:]))
        if k == 1:
            X_train, Y_train, Z_train = X_test, Y_test, Z_test

        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)
        DataLoader_xyz = torch.utils.data.DataLoader(train_xyz, batch_size=batch_size, shuffle=True)

        if not using_orcale:
            current_epochs = epochs_num if k_fold == 0 else max(10, epochs_num // 5)
            G_zy, G_zx = train_ver3(
                G_zx=G_zx, G_zy=G_zy,
                X=X_train, Y=Y_train, Z=Z_train, M=M,
                X_test=X_test, Y_test=Y_test, Z_test=Z_test,
                noise_dimension=noise_dimension, noise_type=noise_dimension_type,
                G_lr=G_lr, hidden_layer_size=hidden_layer_size,
                DataLoader=DataLoader_xyz, BN_type=False, ReLU_coef=0.1,
                lambda_mean=lambda_mean, mean_samples=mean_samples,
                epochs_num=epochs_num, sigma_z=sigma_w, sigma_x=sigma_v, sigma_y=sigma_u,
                normal_ini=normal_ini, lambda_1=lambda_1, lambda_2=lambda_2,
                using_Gen=using_Gen, wgt_decay=wgt_decay, lambda_3=lambda_3,
                drop_out_p=drop_out_p, noise_dimension_var=noise_dimension_var,
                lambda_4=lambda_4,
                device=device)

        dataset_test = DatasetSelect(X_test, Y_test, Z_test)
        dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=True)

        gen_x_all = torch.zeros(test_size, M, device=device)
        gen_y_all = torch.zeros(test_size, M, device=device)
        z_all = torch.zeros(test_size, z_dim, device=device)
        x_all = torch.zeros(test_size, x_dims, device=device)
        y_all = torch.zeros(test_size, y_dims, device=device)
        cur_itr = 0
        G_zx = G_zx.eval()
        G_zy = G_zy.eval()

        for i, (x_test, y_test, z_test) in enumerate(dataloader_test):
            z_test_temp = z_test.repeat(M, 1)
            if not using_orcale:
                z_test_temp = z_test_temp.to(device)
                Noise_fake = sample_noise(z_test_temp.size()[0], noise_dimension,
                                          "normal", device=device).to(device)
                with torch.no_grad():
                    fake_x = G_zx(torch.cat((z_test_temp, Noise_fake), dim=1)).reshape(1, -1)
                Noise_fake = sample_noise(z_test_temp.size()[0], noise_dimension,
                                          "normal", device=device).to(device)
                with torch.no_grad():
                    fake_y = G_zy(torch.cat((z_test_temp, Noise_fake), dim=1)).reshape(1, -1)
            elif using_orcale:
                if simulation == 'type1error':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp, size=M, sType='H0', dx=x_dims, dy=y_dims,
                        dz=z_dim, nstd=nstd, alpha_x=a_x, dist_z=z_dist)
                elif simulation == 'power':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp, size=M, sType='H1', dx=x_dims, dy=y_dims,
                        dz=z_dim, nstd=nstd, alpha_x=a_x, dist_z=z_dist)

            gen_x_all[cur_itr, :] = fake_x.detach().reshape(-1)
            gen_y_all[cur_itr, :] = fake_y.detach().reshape(-1)
            x_all[cur_itr, :] = x_test
            y_all[cur_itr, :] = y_test
            z_all[cur_itr, :] = z_test
            cur_itr = cur_itr + 1

        cur_stat, cur_boot_temp = get_p_value_stat_1(
            boot_num, M, test_size,
            gen_x_all.to(device), gen_y_all.to(device),
            x_all.to(device), y_all.to(device), z_all.to(device),
            sigma_w, sigma_u, sigma_v, boor_rv_type,
            device=device)

        stat_all[cur_k, :] = cur_stat
        boot_temp_all[cur_k, :] = torch.from_numpy(cur_boot_temp)
        cur_k = cur_k + 1

    if using_orcale:
        gen_x_mean = torch.mean(gen_x_all, dim=1).reshape(-1, 1)
        gen_y_mean = torch.mean(gen_y_all, dim=1).reshape(-1, 1)
        mse_x = torch.mean((gen_x_mean - x_all) ** 2).item()
        mse_y = torch.mean((gen_y_mean - y_all) ** 2).item()
        print(f'Test MSE x [{mse_x}], MSE y [{mse_y}]')

    return np.mean(torch.mean(boot_temp_all, dim=0).numpy() > torch.mean(stat_all).item())


# ============================================================
# 并行部分
# ============================================================
from joblib import Parallel, delayed
import multiprocessing
import numpy as np
from datetime import datetime


def run_experiment(params):
    test                 = params["test"]
    sample_size          = params["sample_size"]
    batch_size           = params["batch_size"]
    z_dim                = params["z_dim"]
    dx                   = params["dx"]
    dy                   = params["dy"]
    n_test               = params["n_test"]
    epochs_num           = params["epochs_num"]
    eps_std              = params["eps_std"]
    dist_z               = params["dist_z"]
    alpha_x              = params["alpha_x"]
    m_value              = params["m_value"]
    k_value              = params["k_value"]
    j_value              = params["j_value"]
    noise_dimension      = params["noise_dimension"]
    noise_dimension_type = params["noise_dimension_type"]
    noise_dimension_var  = params["noise_dimension_var"]
    hidden_layer_size    = params["hidden_layer_size"]
    normal_ini           = params["normal_ini"]
    preprocess           = params["preprocess"]
    G_lr                 = params["G_lr"]
    alpha                = params["alpha"]
    alpha1               = params["alpha1"]
    set_seeds            = params["set_seeds"]
    using_orcale         = params["using_orcale"]
    lambda_1             = params["lambda_1"]
    lambda_2             = params["lambda_2"]
    using_Gen            = params["using_Gen"]
    boor_rv_type         = params["boor_rv_type"]
    wgt_decay            = params["wgt_decay"]
    lambda_3             = params["lambda_3"]
    lambda_4             = params["lambda_4"]
    drop_out_p           = params["drop_out_p"]
    lambda_mean          = params["lambda_mean"]
    mean_samples         = params["mean_samples"]

    np.random.seed(set_seeds)
    torch.manual_seed(set_seeds)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(set_seeds)

    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    num_cores = min(32, n_test)
    if num_cores < 1:
        num_cores = 1

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 开始并行实验...")
    print(f"模式: {test}, 样本量: {sample_size}, 交叉验证折数: {k_value}, "
          f"模型均值对齐超参数: {lambda_mean}, 实验次数: {n_test}, "
          f"并行核数: {num_cores}, 可用GPU数: {num_gpus}")
    if test == 'power':
        print(f"备择假设H_1下的模型参数 alpha_x: {alpha_x}")

    p_values = Parallel(n_jobs=num_cores)(
        delayed(mGAN)(
            n=sample_size, z_dim=z_dim, simulation=test, batch_size=batch_size,
            epochs_num=epochs_num, nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
            a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
            noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
            normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
            using_orcale=using_orcale, lambda_1=lambda_1, lambda_2=lambda_2,
            lambda_mean=lambda_mean, mean_samples=mean_samples,
            using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
            lambda_3=lambda_3, drop_out_p=drop_out_p,
            noise_dimension_var=noise_dimension_var,
            noise_dimension_type=noise_dimension_type, lambda_4=lambda_4,
            gpu_id=job_index % num_gpus
        )
        for job_index, _ in enumerate(range(n_test))
    )

    p_values = np.array(p_values)

    fp = [pval < alpha for pval in p_values]
    final_result = np.mean(fp)
    fp1 = [pval < alpha1 for pval in p_values]
    final_result1 = np.mean(fp1)

    print(f"\n" + "=" * 50)
    print(f"实验类型: {test.upper()}")
    print(f"Z Dimension: {z_dim}")
    print(f"Emp Rej Rate: {final_result:.4f} (at significance level {alpha})")
    print(f"Emp Rej Rate: {final_result1:.4f} (at significance level {alpha1})")
    print("=" * 50 + "\n")

    return p_values

In [6]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0.3
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[21:46:03] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 200, 并行核数: 32, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1950 (at significance level 0.1)
Emp Rej Rate: 0.1350 (at significance level 0.05)

[21:48:35] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.0800 (at significance level 0.03490000000000001)
Emp Rej Rate: 0.0500 (at significance level 0.0179)

[21:50:55] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2300 (at significance level 0.03490000000000001)
Emp Rej Rate: 0.1750 (at significance level 0.0179)

[21:53:17] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.3, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.4750 (at significance level 0.0349000000000

In [8]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0.2
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[22:01:10] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.2, 实验次数: 200, 并行核数: 32, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1600 (at significance level 0.1)
Emp Rej Rate: 0.0750 (at significance level 0.05)

[22:04:09] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.2, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1700 (at significance level 0.06770000000000001)
Emp Rej Rate: 0.0900 (at significance level 0.03460000000000001)

[22:07:06] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.2, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2950 (at significance level 0.06770000000000001)
Emp Rej Rate: 0.2250 (at significance level 0.03460000000000001)

[22:10:03] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.2, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.6050 (at signific

In [29]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[17:12:01] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 200, 并行核数: 32, 可用GPU数: 4


Process LokyProcess-2870:
Process LokyProcess-2871:
Process LokyProcess-2864:
Process LokyProcess-2865:
Process LokyProcess-2856:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py", line 528, in _process_worker
    with worker_exit_lock:
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py", line 528, in _process_worker
    with worker_exit_lo

KeyboardInterrupt: 

In [31]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0.5
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[17:14:08] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 200, 并行核数: 32, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1950 (at significance level 0.1)
Emp Rej Rate: 0.1450 (at significance level 0.05)

[17:16:21] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1500 (at significance level 0.0388)
Emp Rej Rate: 0.0700 (at significance level 0.010900000000000002)

[17:18:23] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2750 (at significance level 0.0388)
Emp Rej Rate: 0.1650 (at significance level 0.010900000000000002)

[17:20:20] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0.5, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.5450 (at significance level 0.0388)
Emp R

In [17]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_mean"] = 0
param["mean_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
    param["alpha_x"] = alpha_x
    p_val_list_h1 = run_experiment(param)

[15:18:15] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 200, 并行核数: 32, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1100 (at significance level 0.1)
Emp Rej Rate: 0.0500 (at significance level 0.05)

[15:21:24] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1500 (at significance level 0.0826)
Emp Rej Rate: 0.1100 (at significance level 0.0509)

[15:24:33] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2350 (at significance level 0.0826)
Emp Rej Rate: 0.1500 (at significance level 0.0509)

[15:27:42] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型均值对齐超参数: 0, 实验次数: 200, 并行核数: 32, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.4950 (at significance level 0.0826)
Emp Rej Rate: 0.3950 (at significance lev

In [3]:
# ============================================================
# 参数配置
# ============================================================
param = {
    "test": "power",
    "sample_size": 200,
    "batch_size": 128,
    "z_dim": 1,
    "dx": 1,
    "dy": 1,
    "n_test": 100,
    "epochs_num": 100,
    "eps_std": 1.0,
    "dist_z": 'gaussian',
    "alpha_x": 0.20,
    "m_value": 100,
    "k_value": 8,
    "j_value": 1000,
    "noise_dimension": 5,
    "noise_dimension_type": "normal",
    "noise_dimension_var": 1,
    "hidden_layer_size": 1024,
    "normal_ini": False,
    "preprocess": 'scale',
    "G_lr": 7e-5,
    "alpha": 0.1,
    "alpha1": 0.05,
    "set_seeds": 0,
    "using_orcale": False,
    "lambda_1": 1,
    "lambda_2": 1,
    "using_Gen": '1',
    "boor_rv_type": 'gaussian',
    "wgt_decay": 1e-5,
    "lambda_3": 1e-4,
    "lambda_4": 2e-5,
    "drop_out_p": 0.2,
    "is_sparse": True,
    "sparse_ratio": 0.05,
    "lambda_median": 0.3,
    "median_samples": 30
}

import torch
import torch.distributions as TD
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import numpy as np
from datetime import datetime
import functools
from tqdm import tqdm

# [修改1] 删除全局 device，改为在 mGAN() 内部按 gpu_id 动态创建
# 原代码：
#   enable_cuda = True
#   device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')


def generate_samples_random(size=1000, sType='H0', dx=1, dy=1, dz=1, nstd=1.0, alpha_x=0.5,
                             preprocess="None", dist_z='gaussian', delta_peak=3.0):
    num = size
    error_generator_x = TD.MultivariateNormal(torch.zeros(dx), torch.eye(dx))
    error_generator_y = TD.MultivariateNormal(torch.zeros(dy), torch.eye(dy))

    if dist_z == 'gaussian':
        z_generator = TD.MultivariateNormal(torch.zeros(dz), torch.eye(dz))
    elif dist_z == 'laplace':
        z_generator = TD.Independent(TD.Laplace(torch.zeros(dz), torch.ones(dz)), 1)
    else:
        z_generator = TD.MultivariateNormal(torch.zeros(dz), torch.eye(dz))

    Z = z_generator.sample((num,))
    error_x = error_generator_x.sample((num,))
    error_y = error_generator_y.sample((num,))

    z_abs_mean = torch.mean(torch.abs(Z), dim=1, keepdim=True)
    sigma_z = 0.3 + 1.2 * z_abs_mean

    if dy == dz:
        Z_for_Y = Z
    elif dy < dz:
        Z_for_Y = Z[:, :dy]
    else:
        pad = torch.zeros(num, dy - dz, dtype=Z.dtype, device=Z.device)
        Z_for_Y = torch.cat([Z, pad], dim=1)

    Y = Z_for_Y + nstd * error_y

    if dx == dy:
        error_y_for_x = error_y
    elif dx < dy:
        error_y_for_x = error_y[:, :dx]
    else:
        pad = torch.zeros(num, dx - dy, dtype=error_y.dtype, device=error_y.device)
        error_y_for_x = torch.cat([error_y, pad], dim=1)

    sigma_x = sigma_z.repeat(1, dx)

    if sType == 'H0':
        X = sigma_x * error_x
    else:
        m = TD.Bernoulli(torch.tensor([alpha_x], dtype=Z.dtype, device=Z.device))
        Delta = m.sample((num,)).view(num, 1)
        shared_noise = (1 - Delta) * error_x + Delta * error_y_for_x
        X = sigma_x * shared_noise

    return X, Y, Z


def generate_samples_from_fixed_Z_random(Z, size=1000, sType='H0', dx=1, dy=1, dz=1, nstd=1.0,
                                          alpha_x=0.05, normalize=True, seed=None,
                                          dist_z='gaussian', delta_peak=3.0):
    if seed is not None:
        torch.manual_seed(seed)

    num = size
    if Z.dim() == 1:
        Z = Z.view(num, 1)

    error_generator_x = TD.MultivariateNormal(torch.zeros(dx), torch.eye(dx))
    error_generator_y = TD.MultivariateNormal(torch.zeros(dy), torch.eye(dy))

    error_x = error_generator_x.sample((num,))
    error_y = error_generator_y.sample((num,))

    z_abs_mean = torch.mean(torch.abs(Z), dim=1, keepdim=True)
    sigma_z = 0.3 + 1.2 * z_abs_mean

    if dy == dz:
        Z_for_Y = Z
    elif dy < dz:
        Z_for_Y = Z[:, :dy]
    else:
        pad = torch.zeros(num, dy - dz, dtype=Z.dtype, device=Z.device)
        Z_for_Y = torch.cat([Z, pad], dim=1)

    Y = Z_for_Y + nstd * error_y

    if dx == dy:
        error_y_for_x = error_y
    elif dx < dy:
        error_y_for_x = error_y[:, :dx]
    else:
        pad = torch.zeros(num, dx - dy, dtype=error_y.dtype, device=error_y.device)
        error_y_for_x = torch.cat([error_y, pad], dim=1)

    sigma_x = sigma_z.repeat(1, dx)

    if sType == 'H0':
        X = sigma_x * error_x
    else:
        m = TD.Bernoulli(torch.tensor([alpha_x], dtype=Z.dtype, device=Z.device))
        Delta = m.sample((num,)).view(num, 1)
        shared_noise = (1 - Delta) * error_x + Delta * error_y_for_x
        X = sigma_x * shared_noise

    return X, Y


def get_p_value_stat_1(boot_num, M, n, gen_x_all_torch, gen_y_all_torch, x_torch, y_torch, z_torch,
                        sigma_w, sigma_u=1, sigma_v=1, boor_rv_type="gaussian",
                        device=None):  # [修改2] 新增 device 参数
    w_mx = torch.linalg.vector_norm(
        z_torch.repeat(n, 1, 1) - torch.swapaxes(z_torch.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    w_mx = torch.exp(-w_mx / sigma_w)
    u_mx_1 = torch.exp(-torch.abs(y_torch.repeat(1, n) - y_torch.repeat(1, n).T) / sigma_u)
    u_mx_2 = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch.repeat(n, 1, 1) - y_torch.repeat(1, n).reshape(n, n, 1)) / sigma_u),
        dim=2)
    u_mx_3 = u_mx_2.T
    gen_y_all_torch_rep = gen_y_all_torch.repeat(n, 1, 1)
    temp_mx = gen_y_all_torch_rep[:, :, 0].T
    sum_mx = torch.mean(
        torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)
    v_mx_1 = torch.exp(-torch.abs(x_torch.repeat(1, n) - x_torch.repeat(1, n).T) / sigma_v)
    v_mx_2 = torch.mean(
        torch.exp(-torch.abs(gen_x_all_torch.repeat(n, 1, 1) - x_torch.repeat(1, n).reshape(n, n, 1)) / sigma_v),
        dim=2)
    v_mx_3 = v_mx_2.T
    gen_x_all_torch_rep = gen_x_all_torch.repeat(n, 1, 1)
    temp2_mx = gen_x_all_torch_rep[:, :, 0].T
    sum2_mx = torch.mean(
        torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)
    for i in range(1, M):
        temp_mx = gen_y_all_torch_rep[:, :, i].T
        temp_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_y_all_torch_rep - temp_mx.reshape(n, n, 1)) / sigma_u), dim=2)
        sum_mx = sum_mx + temp_add_mx
        temp2_mx = gen_x_all_torch_rep[:, :, i].T
        temp2_add_mx = torch.mean(
            torch.exp(-torch.abs(gen_x_all_torch_rep - temp2_mx.reshape(n, n, 1)) / sigma_v), dim=2)
        sum2_mx = sum2_mx + temp2_add_mx
    u_mx_4 = 1 / M * sum_mx
    u_mx = u_mx_1 - u_mx_2 - u_mx_3 + u_mx_4
    v_mx_4 = 1 / M * sum2_mx
    v_mx = v_mx_1 - v_mx_2 - v_mx_3 + v_mx_4
    FF_mx = u_mx * v_mx * w_mx * (1 - torch.eye(n).to(device))  # [修改2]
    stat = 1 / (n - 1) * torch.sum(FF_mx).item()
    boottemp = np.array([])
    if boor_rv_type == "rademacher":
        eboot = torch.sign(torch.randn(n, boot_num)).to(device)  # [修改2]
    elif boor_rv_type == "gaussian":
        eboot = torch.randn(n, boot_num).to(device)              # [修改2]
    for bb in range(boot_num):
        random_mx = torch.matmul(eboot[:, bb].reshape(-1, 1), eboot[:, bb].reshape(-1, 1).T)
        bootmatrix = FF_mx * random_mx
        stat_boot = 1 / (n - 1) * torch.sum(bootmatrix).item()
        boottemp = np.append(boottemp, stat_boot)
    return stat, boottemp


class DatasetSelect(Dataset):
    def __init__(self, X, Y, Z):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index]


class DatasetSelect_GAN(torch.utils.data.Dataset):
    def __init__(self, X, Y, Z, batch_size):
        self.X_real = X
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = X.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.X_real[index], self.Y_real[index], self.Z_real[index], \
               self.Z_real[(self.batch_size + index) % self.sample_size]


class DatasetSelect_GAN_ver2(torch.utils.data.Dataset):
    def __init__(self, Y, Z, batch_size):
        self.Y_real = Y
        self.Z_real = Z
        self.batch_size = batch_size
        self.sample_size = Z.shape[0]

    def __len__(self):
        return self.sample_size

    def __getitem__(self, index):
        return self.Y_real[index], self.Z_real[index]


def sample_noise(sample_size, noise_dimension, noise_type, input_var=1,
                 device=None):  # [修改3] 新增 device 参数
    if noise_type == "normal":
        noise_generator = TD.MultivariateNormal(
            torch.zeros(noise_dimension).to(device),           # [修改3]
            input_var * torch.eye(noise_dimension).to(device)) # [修改3]
        Z = noise_generator.sample((sample_size,))
    if noise_type == "unif":
        Z = torch.rand(sample_size, noise_dimension)
    if noise_type == "Cauchy":
        Z = TD.Cauchy(torch.tensor([0.0]), torch.tensor([1.0])).sample(
            (sample_size, noise_dimension)).squeeze(2)
    return Z


class Generator(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension, hidden_layer_size,
                 BN_type, ReLU_coef, drop_out_p, drop_input=False):
        super(Generator, self).__init__()
        self.BN_type = BN_type
        self.ReLU_coef = ReLU_coef
        self.fc1 = torch.nn.Linear(input_dimension + noise_dimension, hidden_layer_size, bias=True)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN2 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
            self.BN3 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)
        self.leakyReLU1 = torch.nn.LeakyReLU(ReLU_coef)
        self.fc2 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc3 = torch.nn.Linear(hidden_layer_size, hidden_layer_size, bias=True)
        self.fc_last = torch.nn.Linear(hidden_layer_size, output_dimension, bias=True)
        self.sigmoid = torch.nn.Sigmoid()
        self.drop_out0 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out1 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out2 = torch.nn.Dropout(p=drop_out_p)
        self.drop_out3 = torch.nn.Dropout(p=drop_out_p)
        self.drop_input = drop_input

    def forward(self, x):
        if self.BN_type:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.BN1(self.fc1(x))))
            x = self.drop_out2(self.leakyReLU1(self.BN2(self.fc2(x))))
            x = self.fc_last(x)
        else:
            if self.drop_input:
                x = self.drop_out0(x)
            x = self.drop_out1(self.leakyReLU1(self.fc1(x)))
            x = self.drop_out2(self.leakyReLU1(self.fc2(x)))
            x = self.fc_last(x)
        return x


class NonFullyConnected_1(torch.nn.Module):
    def __init__(self, size_in, size_out, m, bias=True):
        super(NonFullyConnected_1, self).__init__()
        # [修改3] 不在构造函数中 .to(device)，由外层统一 .to(device)
        self.linear = torch.nn.Linear(m * size_in, m * size_out, bias=bias)
        self.mask = functools.reduce(
            torch.block_diag, [torch.ones(size_out, size_in) for i in range(m)])

    def forward(self, x):
        # [修改3] mask 动态跟随模型所在设备
        self.linear.weight.data *= self.mask.to(self.linear.weight.device)
        return self.linear(x)


class Generator_2(torch.nn.Module):
    def __init__(self, input_dimension, output_dimension, noise_dimension, hidden_layer_size,
                 BN_type, ReLU_coef, hidden_layer_depth=1, ntargets_k=5):
        super(Generator_2, self).__init__()
        self.input_dimension = input_dimension + noise_dimension
        self.output_dimension = output_dimension
        self.ntargets_k = ntargets_k
        self.hidden_layer_sizes = [hidden_layer_size] * hidden_layer_depth
        self.BN_type = BN_type
        self.leakyrelu = torch.nn.LeakyReLU(ReLU_coef)
        self.linear_layers_from_input = torch.nn.Linear(
            self.input_dimension, ntargets_k * self.hidden_layer_sizes[0])
        self.linear_layers_between = torch.nn.ModuleList([
            NonFullyConnected_1(self.hidden_layer_sizes[0], self.hidden_layer_sizes[0], ntargets_k)
            for i in range(len(self.hidden_layer_sizes))
        ])
        self.linear8 = torch.nn.Linear(self.hidden_layer_sizes[0] * ntargets_k, self.output_dimension)
        if BN_type:
            self.BN1 = torch.nn.BatchNorm1d(hidden_layer_size, 0.8, affine=False)

    def forward(self, input):
        if self.BN_type:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(self.BN1(output))
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(self.BN1(output))
        else:
            output = self.linear_layers_from_input(input)
            output = self.leakyrelu(output)
            for linear_layers_between in self.linear_layers_between:
                output = linear_layers_between(output)
                output = self.leakyrelu(output)
        return self.linear8(output)


def find_loss(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-torch.abs(Y.repeat(1, n) - Y.repeat(1, n).T) / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    mx_1_2 = torch.exp(-mx_1_2 / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-torch.abs(Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-torch.abs(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss


def find_loss_2(Y, hat_Y, Z, sigma_z, sigma_y):
    n = Z.shape[0]
    mx_1_1 = torch.exp(-(Y.repeat(1, n) - Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_1_2 = torch.linalg.vector_norm(
        Z.repeat(n, 1, 1) - torch.swapaxes(Z.repeat(n, 1, 1), 0, 1), ord=2, dim=2)
    mx_1_2 = torch.exp(-(mx_1_2 ** 2) / sigma_z)
    mx_1 = mx_1_1 * mx_1_2
    mx_2_1 = torch.exp(-(Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_2 = mx_2_1 * mx_1_2
    mx_3 = mx_2.T
    mx_4_1 = torch.exp(-(hat_Y.repeat(1, n) - hat_Y.repeat(1, n).T) ** 2 / sigma_y)
    mx_4 = mx_4_1 * mx_1_2
    FF_mx = (mx_1 - mx_2 - mx_3 + mx_4)
    loss = 1 / (n ** 2) * torch.sum(FF_mx)
    return loss


def pinball_loss(y_true, y_pred, tau=0.5):
    diff = y_true - y_pred
    loss = torch.where(diff >= 0,
                       tau * diff,
                       (1.0 - tau) * (-diff))
    return loss.mean()


def train_ver3(
        G_zx, G_zy,
        X, Y, Z, X_test, Y_test, Z_test, M,
        noise_dimension, noise_type, G_lr, hidden_layer_size,
        DataLoader, BN_type, ReLU_coef,
        lambda_median=0.5, median_samples=20,
        epochs_num=50,
        patience=5, min_delta=1e-5,
        sigma_z=1, sigma_x=1, sigma_y=1,
        normal_ini=False,
        lambda_1=1, lambda_2=1, using_Gen='1', wgt_decay=0,
        lambda_3=0, drop_out_p=0.2, noise_dimension_var=1,
        lambda_4=0,
        device=None):  # [修改3] 新增 device 参数

    input_dimension = Z.shape[1]
    output_dimension_y = Y.shape[1]
    output_dimension_x = X.shape[1]

    if G_zy is None or G_zx is None:
        if using_Gen == '1':
            G_zy = Generator(input_dimension, output_dimension_y, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改3]
            G_zx = Generator(input_dimension, output_dimension_x, noise_dimension,
                             hidden_layer_size, BN_type, ReLU_coef, drop_out_p).to(device)  # [修改3]
        elif using_Gen == '2':
            G_zy = Generator_2(input_dimension, output_dimension_y, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改3]
            G_zx = Generator_2(input_dimension, output_dimension_x, noise_dimension,
                               hidden_layer_size, BN_type, ReLU_coef).to(device)             # [修改3]

    if normal_ini:
        for p in G_zy.parameters():
            p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改3]
        for p in G_zx.parameters():
            p.data = torch.randn(p.shape, device=device) / np.sqrt(float(hidden_layer_size * 2))  # [修改3]

    G_zy_solver = optim.Adam(G_zy.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)
    G_zx_solver = optim.Adam(G_zx.parameters(), lr=G_lr, betas=(0.5, 0.999), weight_decay=wgt_decay)

    iter_count = 0
    G_zy = G_zy.train()
    G_zx = G_zx.train()

    best_loss = float('inf')
    counter = 0

    for epoch in range(epochs_num):
        batch_count = 0
        G_zy = G_zy.train()
        G_zx = G_zx.train()

        for X_real, Y_real, Z_real, Z_fake in DataLoader:
            X_real = X_real.to(device)   # [修改3]
            Y_real = Y_real.to(device)   # [修改3]
            Z_real = Z_real.to(device)   # [修改3]
            Z_fake = Z_fake.to(device)   # [修改3]

            batch_size = Z_real.shape[0]
            Z_repeated = Z_real.repeat_interleave(median_samples, dim=0)

            Noise_for_median = sample_noise(
                Z_repeated.shape[0], noise_dimension, noise_type,
                input_var=noise_dimension_var, device=device).to(device)  # [修改3]

            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)  # [修改3]
            Y_fake = G_zy(torch.cat((Z_real, Noise_fake), dim=1)).to(device)
            Noise_fake = sample_noise(Z_real.shape[0], noise_dimension, noise_type,
                                      input_var=noise_dimension_var, device=device).to(device)  # [修改3]
            X_fake = G_zx(torch.cat((Z_real, Noise_fake), dim=1)).to(device)

            Y_generated_group = G_zy(torch.cat((Z_repeated, Noise_for_median), dim=1))
            Y_generated_reshaped = Y_generated_group.reshape(batch_size, median_samples, -1)
            Y_median_pred = torch.median(Y_generated_reshaped, dim=1).values
            loss_median_y = pinball_loss(Y_real, Y_median_pred, tau=0.5)

            X_generated_group = G_zx(torch.cat((Z_repeated, Noise_for_median), dim=1))
            X_generated_reshaped = X_generated_group.reshape(batch_size, median_samples, -1)
            X_median_pred = torch.median(X_generated_reshaped, dim=1).values
            loss_median_x = pinball_loss(X_real, X_median_pred, tau=0.5)

            g_zy_error = None
            G_zy_solver.zero_grad()
            g_zx_error = None
            G_zx_solver.zero_grad()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param_g in G_zy.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param_g, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param_g, ord=1)

            mmd_loss_y = (lambda_1 * find_loss(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_2 * find_loss_2(Y_real, Y_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_y) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zy_error = mmd_loss_y + lambda_median * loss_median_y
            g_zy_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zy.parameters(), max_norm=0.5)
            G_zy_solver.step()

            l1_regularization_first_layer = 0
            l1_regularization_rest_layers = 0
            for name, param_g in G_zx.named_parameters():
                if "fc1" in name:
                    l1_regularization_first_layer += torch.linalg.vector_norm(param_g, ord=1)
                else:
                    l1_regularization_rest_layers += torch.linalg.vector_norm(param_g, ord=1)

            mmd_loss_x = (lambda_1 * find_loss(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_2 * find_loss_2(X_real, X_fake, Z_real, sigma_z=sigma_z, sigma_y=sigma_x) +
                          lambda_3 * l1_regularization_rest_layers +
                          lambda_4 * l1_regularization_first_layer)
            g_zx_error = mmd_loss_x + lambda_median * loss_median_x
            g_zx_error.backward()
            torch.nn.utils.clip_grad_norm_(G_zx.parameters(), max_norm=0.5)
            G_zx_solver.step()

            iter_count += 1
            batch_count += 1

        current_loss = g_zx_error + g_zy_error
        if current_loss < best_loss - min_delta:
            best_loss = current_loss
            counter = 0
        else:
            counter += 1
        if counter >= patience:
            break

    return G_zy, G_zx


def mGAN(n=500, z_dim=1, simulation='type1error', batch_size=64, epochs_num=1000,
         nstd=1.0, z_dist='gaussian', x_dims=1, y_dims=1, a_x=0.05, M=500, k=2, boot_num=1000,
         noise_dimension=10, hidden_layer_size=512, normal_ini=False, preprocess='normalize',
         G_lr=1e-5, using_orcale=False, lambda_1=1, lambda_2=1,
         lambda_median=0.3, median_samples=20,
         using_Gen='1',
         boor_rv_type="gaussian", wgt_decay=0, lambda_3=1, drop_out_p=0.2,
         noise_dimension_var=1, noise_dimension_type="normal", lambda_4=1,
         gpu_id=0):  # [修改4] 新增 gpu_id 参数

    # [修改4] 在函数内部根据 gpu_id 创建局部 device
    enable_cuda = True
    if torch.cuda.is_available() and enable_cuda:
        device = torch.device(f'cuda:{gpu_id}')
    else:
        device = torch.device('cpu')

    if simulation == 'type1error':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n, sType='H0', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess)
    elif simulation == 'power':
        sim_x, sim_y, sim_z = generate_samples_random(
            size=n, sType='H1', dx=x_dims, dy=y_dims, dz=z_dim,
            nstd=nstd, alpha_x=a_x, dist_z=z_dist, preprocess=preprocess)
    else:
        raise ValueError('Test does not exist.')

    x, y, z = sim_x.to(device), sim_y.to(device), sim_z.to(device)

    w_mx = torch.linalg.vector_norm(
        z.repeat(n, 1, 1) - torch.swapaxes(z.repeat(n, 1, 1), 0, 1), ord=1, dim=2)
    sigma_w = torch.median(w_mx).item()
    u_mx = torch.exp(-torch.abs(y.repeat(1, n) - y.repeat(1, n).T))
    sigma_u = torch.median(torch.abs(y.repeat(1, n) - y.repeat(1, n).T)).item()
    v_mx = torch.exp(-torch.abs(x.repeat(1, n) - x.repeat(1, n).T))
    sigma_v = torch.median(torch.abs(x.repeat(1, n) - x.repeat(1, n).T)).item()

    test_size = int(n / k)
    stat_all = torch.zeros(k, 1)
    boot_temp_all = torch.zeros(k, boot_num)
    cur_k = 0

    if not using_orcale:
        G_zy = Generator(z_dim, y_dims, noise_dimension, hidden_layer_size, False, 0.1, drop_out_p).to(device)
        G_zx = Generator(z_dim, x_dims, noise_dimension, hidden_layer_size, False, 0.1, drop_out_p).to(device)

    for k_fold in range(k):
        k_fold_start = int(n / k * k_fold)
        k_fold_end = int(n / k * (k_fold + 1))
        X_test = x[k_fold_start:k_fold_end]
        Y_test = y[k_fold_start:k_fold_end]
        Z_test = z[k_fold_start:k_fold_end]
        X_train = torch.cat((x[0:k_fold_start], x[k_fold_end:]))
        Y_train = torch.cat((y[0:k_fold_start], y[k_fold_end:]))
        Z_train = torch.cat((z[0:k_fold_start], z[k_fold_end:]))
        if k == 1:
            X_train, Y_train, Z_train = X_test, Y_test, Z_test

        train_xyz = DatasetSelect_GAN(X_train, Y_train, Z_train, batch_size)
        DataLoader_xyz = torch.utils.data.DataLoader(train_xyz, batch_size=batch_size, shuffle=True)

        if not using_orcale:
            current_epochs = epochs_num if k_fold == 0 else max(10, epochs_num // 5)
            G_zy, G_zx = train_ver3(
                G_zx=G_zx, G_zy=G_zy,
                X=X_train, Y=Y_train, Z=Z_train, M=M,
                X_test=X_test, Y_test=Y_test, Z_test=Z_test,
                noise_dimension=noise_dimension, noise_type=noise_dimension_type,
                G_lr=G_lr, hidden_layer_size=hidden_layer_size,
                DataLoader=DataLoader_xyz, BN_type=False, ReLU_coef=0.1,
                lambda_median=lambda_median, median_samples=median_samples,
                epochs_num=epochs_num,
                sigma_z=sigma_w, sigma_x=sigma_v, sigma_y=sigma_u,
                normal_ini=normal_ini, lambda_1=lambda_1, lambda_2=lambda_2,
                using_Gen=using_Gen, wgt_decay=wgt_decay, lambda_3=lambda_3,
                drop_out_p=drop_out_p, noise_dimension_var=noise_dimension_var,
                lambda_4=lambda_4,
                device=device)  # [修改4] 传入局部 device

        dataset_test = DatasetSelect(X_test, Y_test, Z_test)
        dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=True)

        gen_x_all = torch.zeros(test_size, M, device=device)   # [修改4]
        gen_y_all = torch.zeros(test_size, M, device=device)   # [修改4]
        z_all = torch.zeros(test_size, z_dim, device=device)   # [修改4]
        x_all = torch.zeros(test_size, x_dims, device=device)  # [修改4]
        y_all = torch.zeros(test_size, y_dims, device=device)  # [修改4]
        cur_itr = 0

        if not using_orcale:
            G_zx = G_zx.eval()
            G_zy = G_zy.eval()

        for i, (x_test, y_test, z_test) in enumerate(dataloader_test):
            z_test_temp = z_test.repeat(M, 1)
            if not using_orcale:
                z_test_temp = z_test_temp.to(device)                                      # [修改4]
                Noise_fake = sample_noise(z_test_temp.size()[0], noise_dimension,
                                          "normal", device=device).to(device)              # [修改4]
                with torch.no_grad():
                    fake_x = G_zx(torch.cat((z_test_temp, Noise_fake), dim=1)).reshape(1, -1)
                Noise_fake = sample_noise(z_test_temp.size()[0], noise_dimension,
                                          "normal", device=device).to(device)              # [修改4]
                with torch.no_grad():
                    fake_y = G_zy(torch.cat((z_test_temp, Noise_fake), dim=1)).reshape(1, -1)
            elif using_orcale:
                if simulation == 'type1error':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp, size=M, sType='H0', dx=x_dims, dy=y_dims,
                        dz=z_dim, nstd=nstd, alpha_x=a_x, dist_z=z_dist)
                elif simulation == 'power':
                    fake_x, fake_y = generate_samples_from_fixed_Z_random(
                        z_test_temp, size=M, sType='H1', dx=x_dims, dy=y_dims,
                        dz=z_dim, nstd=nstd, alpha_x=a_x, dist_z=z_dist)

            gen_x_all[cur_itr, :] = fake_x.detach().reshape(-1)
            gen_y_all[cur_itr, :] = fake_y.detach().reshape(-1)
            x_all[cur_itr, :] = x_test
            y_all[cur_itr, :] = y_test
            z_all[cur_itr, :] = z_test
            cur_itr = cur_itr + 1

        cur_stat, cur_boot_temp = get_p_value_stat_1(
            boot_num, M, test_size,
            gen_x_all.to(device), gen_y_all.to(device),
            x_all.to(device), y_all.to(device), z_all.to(device),
            sigma_w, sigma_u, sigma_v, boor_rv_type,
            device=device)  # [修改4] 传入局部 device
        stat_all[cur_k, :] = cur_stat
        boot_temp_all[cur_k, :] = torch.from_numpy(cur_boot_temp)
        cur_k = cur_k + 1

    if using_orcale:
        gen_x_median = torch.median(gen_x_all, dim=1).values.reshape(-1, 1)
        gen_y_median = torch.median(gen_y_all, dim=1).values.reshape(-1, 1)
        mse_x = torch.mean((gen_x_median - x_all) ** 2).item()
        mse_y = torch.mean((gen_y_median - y_all) ** 2).item()
        print(f'Test MSE x (median diff) [{mse_x}], MSE y (median diff) [{mse_y}]')

    return np.mean(torch.mean(boot_temp_all, dim=0).numpy() > torch.mean(stat_all).item())


# ============================================================
# 并行部分
# ============================================================
from joblib import Parallel, delayed
import multiprocessing

def run_experiment(params):
    test                 = params["test"]
    sample_size          = params["sample_size"]
    batch_size           = params["batch_size"]
    z_dim                = params["z_dim"]
    dx                   = params["dx"]
    dy                   = params["dy"]
    n_test               = params["n_test"]
    epochs_num           = params["epochs_num"]
    eps_std              = params["eps_std"]
    dist_z               = params["dist_z"]
    alpha_x              = params["alpha_x"]
    m_value              = params["m_value"]
    k_value              = params["k_value"]
    j_value              = params["j_value"]
    noise_dimension      = params["noise_dimension"]
    noise_dimension_type = params["noise_dimension_type"]
    noise_dimension_var  = params["noise_dimension_var"]
    hidden_layer_size    = params["hidden_layer_size"]
    normal_ini           = params["normal_ini"]
    preprocess           = params["preprocess"]
    G_lr                 = params["G_lr"]
    alpha                = params["alpha"]
    alpha1               = params["alpha1"]
    set_seeds            = params["set_seeds"]
    using_orcale         = params["using_orcale"]
    lambda_1             = params["lambda_1"]
    lambda_2             = params["lambda_2"]
    using_Gen            = params["using_Gen"]
    boor_rv_type         = params["boor_rv_type"]
    wgt_decay            = params["wgt_decay"]
    lambda_3             = params["lambda_3"]
    lambda_4             = params["lambda_4"]
    drop_out_p           = params["drop_out_p"]
    lambda_median        = params["lambda_median"]
    median_samples       = params["median_samples"]

    np.random.seed(set_seeds)
    torch.manual_seed(set_seeds)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(set_seeds)

    # [修改4] 获取可用 GPU 数量，控制并发数，轮询分配 GPU
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    num_cores = min(28, n_test)
    if num_cores < 1:
        num_cores = 1

    print(f"[{datetime.now().strftime('%H:%M:%S')}] 开始并行实验...")
    print(f"模式: {test}, 样本量: {sample_size}, 交叉验证折数: {k_value}, "
          f"模型中位数对齐超参数(pinball): {lambda_median}, 实验次数: {n_test}, "
          f"并行核数: {num_cores}, 可用GPU数: {num_gpus}")
    if test == 'power':
        print(f"备择假设H_1下的模型参数 alpha_x: {alpha_x}")

    # [修改4] 按 job_index % num_gpus 轮询分配 GPU
    p_values = Parallel(n_jobs=num_cores)(
        delayed(mGAN)(
            n=sample_size, z_dim=z_dim, simulation=test, batch_size=batch_size,
            epochs_num=epochs_num, nstd=eps_std, z_dist=dist_z, x_dims=dx, y_dims=dy,
            a_x=alpha_x, M=m_value, k=k_value, boot_num=j_value,
            noise_dimension=noise_dimension, hidden_layer_size=hidden_layer_size,
            normal_ini=normal_ini, preprocess=preprocess, G_lr=G_lr,
            using_orcale=using_orcale, lambda_1=lambda_1, lambda_2=lambda_2,
            lambda_median=lambda_median, median_samples=median_samples,
            using_Gen=using_Gen, boor_rv_type=boor_rv_type, wgt_decay=wgt_decay,
            lambda_3=lambda_3, drop_out_p=drop_out_p,
            noise_dimension_var=noise_dimension_var,
            noise_dimension_type=noise_dimension_type, lambda_4=lambda_4,
            gpu_id=job_index % num_gpus  # [修改4] 轮询分配 GPU
        )
        for job_index, _ in enumerate(range(n_test))  # [修改4] enumerate 获取 job_index
    )

    p_values = np.array(p_values)
    fp = [pval < alpha for pval in p_values]
    final_result = np.mean(fp)
    fp1 = [pval < alpha1 for pval in p_values]
    final_result1 = np.mean(fp1)

    print(f"\n" + "=" * 50)
    print(f"实验类型: {test.upper()}")
    print(f"Z Dimension: {z_dim}")
    print(f"Emp Rej Rate: {final_result:.4f} (at significance level {alpha})")
    print(f"Emp Rej Rate: {final_result1:.4f} (at significance level {alpha1})")
    print("=" * 50 + "\n")

    return p_values

In [4]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_median"] = 0
param["median_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[10:48:27] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4


/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1150 (at significance level 0.1)
Emp Rej Rate: 0.0600 (at significance level 0.05)

[10:55:03] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1400 (at significance level 0.08500000000000002)
Emp Rej Rate: 0.0950 (at significance level 0.04070000000000001)

[11:01:22] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.2150 (at significance level 0.08500000000000002)
Emp Rej Rate: 0.1350 (at significance level 0.04070000000000001)

[11:08:15] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.4350 (at significance level 0.08500000000000002)
Emp Rej Rate: 0.2850 (at significance level 0.040

Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/backend/popen_loky_posix.py", line 155, in <module>
    parser = argparse.ArgumentParser("Command line parser")
  File "/opt/anaconda3/lib/python3.9/argparse.py", line 1728, in __init__
    self._positionals = add_group(_('positional arguments'))
  File "/opt/anaconda3/lib/python3.9/gettext.py", line 742, in gettext
    return dgettext(_current_domain, message)
  File "/opt/anaconda3/lib/python3.9/gettext.py", line 666, in dgettext
    t = translation(domain, _localedirs.get(domain, None))
  File "/opt/anaconda3/lib/python3.9/gettext.py", line 587, in translation
    mofiles = find(domain, localedir, languages, all=True)
  File "/opt/anacond

KeyboardInterrupt: 

In [6]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_median"] = 0.5
param["median_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[11:18:58] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1500 (at significance level 0.1)
Emp Rej Rate: 0.0950 (at significance level 0.05)

[11:24:59] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1850 (at significance level 0.051800000000000006)
Emp Rej Rate: 0.1000 (at significance level 0.024950000000000003)

[11:32:59] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3600 (at significance level 0.051800000000000006)
Emp Rej Rate: 0.2300 (at significance level 0.024950000000000003)

[11:38:00] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.5, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Di

In [2]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_median"] = 0.3
param["median_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[20:46:45] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.3, 实验次数: 200, 并行核数: 28, 可用GPU数: 4


/home/23099458d/venvs/myjupyter/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1000 (at significance level 0.1)
Emp Rej Rate: 0.0500 (at significance level 0.05)

[20:51:13] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.3, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1850 (at significance level 0.09950000000000002)
Emp Rej Rate: 0.1100 (at significance level 0.054350000000000016)

[20:55:30] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.3, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3950 (at significance level 0.09950000000000002)
Emp Rej Rate: 0.2600 (at significance level 0.054350000000000016)

[20:59:50] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.3, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.7200 (at significance level 0.09950000000000002)
Emp Rej Rate: 0.6300 (at significance lev

In [4]:
param["sample_size"] = 400
param["k_value"] = 2

# 固定阈值看 type1error
param["test"] = "type1error"
param["n_test"] = 200
param["lambda_median"] = 0.2
param["median_samples"] = 30
p_val_list = run_experiment(param)

# H0 校准 power 用的阈值
alpha_emp_10 = np.quantile(p_val_list, 0.10)
alpha_emp_05 = np.quantile(p_val_list, 0.05)

# 跑 power
param["test"] = "power"
param["alpha"] = alpha_emp_10
param["alpha1"] = alpha_emp_05

for alpha_x in [0.05, 0.10, 0.15, 0.20, 0.25]:
        param["alpha_x"] = alpha_x
        p_val_list_h1 = run_experiment(param)

[21:14:39] 开始并行实验...
模式: type1error, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.2, 实验次数: 200, 并行核数: 28, 可用GPU数: 4

实验类型: TYPE1ERROR
Z Dimension: 1
Emp Rej Rate: 0.1300 (at significance level 0.1)
Emp Rej Rate: 0.0600 (at significance level 0.05)

[21:19:00] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.2, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.05

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.1700 (at significance level 0.07550000000000001)
Emp Rej Rate: 0.1150 (at significance level 0.044700000000000004)

[21:23:22] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.2, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.1

实验类型: POWER
Z Dimension: 1
Emp Rej Rate: 0.3800 (at significance level 0.07550000000000001)
Emp Rej Rate: 0.2900 (at significance level 0.044700000000000004)

[21:27:46] 开始并行实验...
模式: power, 样本量: 400, 交叉验证折数: 2, 模型中位数对齐超参数(pinball): 0.2, 实验次数: 200, 并行核数: 28, 可用GPU数: 4
备择假设H_1下的模型参数 alpha_x: 0.15

实验类型: POWER
Z Dime